# 第 13 章 AD-AS モデルのキャリブレーション: 数字を変えて確かめる

**使い方**: 「Parameters」のセルの数字を書き換えて、上から順に実行します（メニューの Run → Run All Cells）。数字を変えるたびに、結果と図が変わります。

起動: アプリのディレクトリで `uv run --with jupyterlab jupyter lab notebooks/` を実行します。

記事: `docs/article/macro-programming/13-adas-calibration.md`

In [ ]:
# --- Parameters: ここの数字を書き換えます ---
a = 0.8        # AS 曲線の傾き
c = 0.3        # AD 曲線の傾き
sigma_u = 1.5  # 需要ショックの標準偏差
sigma_v = 1.2  # 供給ショックの標準偏差
deviation = 1.0  # 今期の物価の乖離（%）

In [ ]:
# --- 準備（変更しません）: lib を読み込めるようにします ---
import sys
from pathlib import Path

_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "lib").is_dir())
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

## 物価の持続性 h

In [ ]:
from lib.chapter13.adas import AdasParameters

params = AdasParameters(a=a, c=c, sigma_u=sigma_u, sigma_v=sigma_v)
print(f"h = 1 ÷ (1 + a·c) = {params.h:.4f}")
next_deviation = deviation * params.h
print(f"ショックがなければ、{deviation}% の乖離は、次の期に {next_deviation:.4f} %")

## モデルの特徴と、データの特徴

In [ ]:
from lib.chapter13.adas import (
    adas_cycles,
    data_features,
    load_adas_quarterly,
    model_moments,
)

cycles = adas_cycles(load_adas_quarterly())
data = data_features(cycles["y"], cycles["p"])
model = model_moments(params)
for key in data:
    print(f"{key}: データ {data[key]:.3f}、モデル {model[key]:.3f}")

## 図: データとモデルの比較

In [ ]:
from lib.chapter13.plot import plot_moment_comparison

plot_moment_comparison(data, model)

## 試してみよう

- `a = 1.6` にすると、h は 0.8065 から 0.6757 に下がり、
  モデルの物価の自己相関も 0.676 に下がります
- `a = 0.4`、`c = 0.6` にすると、a·c は変わらないので、h は 0.8065 のままで、
  モデルの物価の自己相関も 0.806 のままです。ただし、産出量の標準偏差は 1.608、
  産出量の自己相関は 0.240 に変わります
- `sigma_u = 0.5` にすると、モデルの産出量の標準偏差は 1.364 から 0.649 に下がります